**LSA16 Detector Usability Table (movement + zero rate + failure)**

This section builds a detector mapping table for LD-LSA16 using the Excel mapping information (Detector Name, ID, Movement).
Then it evaluates each detector using September time-series measurements:

Zero rate (count): percentage of timeFrames where count == 0

Controller failure: detectors reported as faulty in the intersection snapshot (hardwareStatus.faultDetectors)

Detectors are labeled:

✅ USABLE: not faulty and not always-zero

❌ NOT USABLE (X): faulty or always-zero (with reason)

In [1]:
from pathlib import Path
import json
from collections import defaultdict

# =========================
# PATHS (adjust if needed)
# =========================
INTERSECTIONS_DIR = Path(
    r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections"
)

# September dataset base folder you gave:
SEP_BASE = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\data\Landau\2025-09")

# September folder name for LSA16 (your dataset uses LDLSA16 etc.)
SEP_FOLDER_LSA16 = SEP_BASE / "LDLSA16"

# =========================
# LSA16 mapping (from your Excel screenshot)
# (DetectorName, DetectorID, Movement, DemandFlag, EdgeRelation)
# =========================
LSA16_MAPPING = [
    # West approach
    ("VD23", 5,  "S+R", "yes", "no"),
    ("WD22", 4,  "L",   "yes", "yes"),
    ("WD2.1", 3, "S",   "no",  "yes"),

    # East approach
    ("VD4.3", 10, "S+R", "yes", "no"),
    ("VD4.4", 11, "L",   "yes", "no"),
    ("WD4.1", 8,  "S+R", "no",  "yes"),
    ("WD4.2", 9,  "L",   "no",  "yes"),

    # North approach
    ("VD3.2", 7, "S",    "yes", "yes"),
    ("VD3.1", 6, "R",    "yes", "yes"),

    # South approach
    ("VD1.3", 33, "L",   "yes", "yes"),
    ("VD1.2", 2,  "S",   "yes", "yes"),
    ("VD1.1", 1,  "R",   "yes", "yes"),
]

# Tuning: define when "almost always zero" becomes unusable
ALMOST_ZERO_THRESHOLD_PCT = 99.0  # if >=99% of frames have count==0, treat as not usable

# =========================
# Common helpers
# =========================
def list_json_files(folder: Path):
    return sorted([p for p in folder.glob("*.json") if p.is_file()])

def load_tlc_by_alias(intersections_dir: Path, alias_target: str):
    for fp in list_json_files(intersections_dir):
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
        tlc = data.get("tlc", {})
        if tlc.get("alias") == alias_target or tlc.get("name") == alias_target:
            return tlc, fp
    return None, None

def extract_fault_detectors(tlc: dict):
    hw = tlc.get("hardwareStatus", {})
    faults = []
    for fd in hw.get("faultDetectors", []) or []:
        fd_id = fd.get("id")
        fd_name = fd.get("name")
        if fd_id is None:
            continue
        faults.append((int(fd_id), fd_name))
    uniq = {}
    for d_id, d_name in faults:
        uniq[d_id] = (d_id, d_name)
    return [uniq[k] for k in sorted(uniq.keys())]

def safe_get_count(det_obj):
    reading = det_obj.get("reading", {})
    c = reading.get("count", {})
    return c.get("value")

def pct(z, n):
    return 0.0 if n == 0 else 100.0 * z / n

# September chunk names look like: 2025-09-01_000000_045959.json
def day_from_filename(fp: Path):
    return fp.name.split("_")[0] if "_" in fp.name else fp.stem

# =========================
# A) Controller failure check (from intersection snapshot)
# =========================
tlc, fp = load_tlc_by_alias(INTERSECTIONS_DIR, "LD-LSA16")
if tlc is None:
    print("WARNING: Could not find LD-LSA16 in intersections folder. Failure check will be skipped.")
    fault_ids = set()
    fault_list = []
else:
    fault_list = extract_fault_detectors(tlc)
    fault_ids = {d_id for d_id, _ in fault_list}

print("\n==============================")
print("LSA16 CONTROLLER FAILURE CHECK")
print("==============================")
if tlc is not None:
    print("Intersection file:", fp)
    print("active:", tlc.get("active"))
    print("state:", tlc.get("state"))
    print("brokenDetectorCount:", tlc.get("brokenDetectorCount"))
    print("lastStatusUpdate:", tlc.get("lastStatusUpdate"))

print("\nController-reported faulty detectors:")
if not fault_list:
    print("(none)")
else:
    for d_id, d_name in fault_list:
        print(f"- ID={d_id} | Name={d_name}")

# =========================
# B) Time-series zero-rate analysis (September chunks)
# =========================
# We aggregate across ALL September files in LDLSA16 folder.
files = list_json_files(SEP_FOLDER_LSA16)
print("\n==============================")
print("LSA16 SEPTEMBER FILES CHECK")
print("==============================")
print("Folder:", SEP_FOLDER_LSA16)
print("Files found:", len(files))

# stats per detector_id
stats = defaultdict(lambda: {"frames": 0, "count_zero": 0, "count_nonzero": 0})

for fp_ts in files:
    with open(fp_ts, "r", encoding="utf-8") as f:
        day_chunk = json.load(f)

    for tf in day_chunk.get("timeFrames", []):
        for det in tf.get("detectors", []):
            det_id = det.get("id")
            if det_id is None:
                continue
            det_id = int(det_id)

            c = safe_get_count(det)
            if not isinstance(c, (int, float)):
                continue

            stats[det_id]["frames"] += 1
            if c == 0:
                stats[det_id]["count_zero"] += 1
            else:
                stats[det_id]["count_nonzero"] += 1

# =========================
# C) Build mapping table + usability decision
# =========================
rows = []
for name, det_id, movement, demand_flag, edge_rel in LSA16_MAPPING:
    s = stats.get(det_id, {"frames": 0, "count_zero": 0, "count_nonzero": 0})
    n = s["frames"]
    cz = s["count_zero"]
    zr = pct(cz, n)

    reasons = []
    usable = True

    # 1) controller failure
    if det_id in fault_ids:
        usable = False
        reasons.append("controller faultDetectors")

    # 2) time-series always/almost-always zero (COUNT)
    if n == 0:
        usable = False
        reasons.append("no timeFrames found in September data")
    else:
        if cz == n and s["count_nonzero"] == 0:
            usable = False
            reasons.append("count always 0 (100%)")
        elif zr >= ALMOST_ZERO_THRESHOLD_PCT:
            usable = False
            reasons.append(f"count almost always 0 (>= {ALMOST_ZERO_THRESHOLD_PCT}%)")

    status = "✅ USABLE" if usable else "❌ X (NOT USABLE)"
    reason_text = " / ".join(reasons) if reasons else "-"

    rows.append({
        "Detector": name,
        "Detector_ID": det_id,
        "Movement": movement,
        "Demand": demand_flag,
        "Edge_relation": edge_rel,
        "Frames": n,
        "Zero_rate_% (count)": round(zr, 1),
        "Status": status,
        "Reason": reason_text
    })

# Pretty print without pandas (stable)
print("\n==============================")
print("LSA16 DETECTOR USABILITY TABLE")
print("==============================")
header = ["Detector","Detector_ID","Movement","Demand","Edge_relation","Frames","Zero_rate_% (count)","Status","Reason"]
print(" | ".join(header))
print("-" * 140)
for r in rows:
    print(
        f"{r['Detector']:<8} | {r['Detector_ID']:<10} | {r['Movement']:<7} | {r['Demand']:<6} | {r['Edge_relation']:<13} | "
        f"{r['Frames']:<6} | {r['Zero_rate_% (count)']:<16} | {r['Status']:<16} | {r['Reason']}"
    )

# =========================
# D) Explain zero rate and usage note
# =========================
print("\n==============================")
print("What does 'Zero rate' mean?")
print("==============================")
print(
    "Zero rate (count) = (# of timeFrames where detector count == 0) / (total # of timeFrames for that detector) * 100.\n"
    "Interpretation:\n"
    "- Low zero rate can be normal (e.g., at night / low demand).\n"
    "- Very high zero rate (near 100%) typically indicates a broken detector, wrong mapping, or missing data.\n"
    f"- In this notebook, detectors with zero rate >= {ALMOST_ZERO_THRESHOLD_PCT}% (or exactly 100%) are marked NOT USABLE."
)

print("\n==============================")
print("Note for demand generation / calibration")
print("==============================")
print(
    "For demand generation (routeSampler/dfrouter) we prefer detectors that are:\n"
    "- marked as Demand=yes in the mapping,\n"
    "- NOT faulty in the controller snapshot,\n"
    "- NOT always-zero in the time-series.\n\n"
    "Detectors marked as '❌ X (NOT USABLE)' should be excluded from demand generation and LSTM inputs.\n"
    "If a detector is Demand=no but still usable, it can be used as an internal consistency check (edge_relation / movement sanity)."
)


LSA16 CONTROLLER FAILURE CHECK
Intersection file: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections\LSA16.json
active: True
state: DETECTOR_FAILURE
brokenDetectorCount: 2
lastStatusUpdate: 2025-06-04T09:30:37

Controller-reported faulty detectors:
- ID=8 | Name=WD41
- ID=9 | Name=WD42

LSA16 SEPTEMBER FILES CHECK
Folder: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\data\Landau\2025-09\LDLSA16
Files found: 87

LSA16 DETECTOR USABILITY TABLE
Detector | Detector_ID | Movement | Demand | Edge_relation | Frames | Zero_rate_% (count) | Status | Reason
--------------------------------------------------------------------------------------------------------------------------------------------
VD23     | 5          | S+R     | yes    | no            | 1740   | 3.0              | ✅ USABLE         | -
WD22     | 4          | L       | yes    | yes           | 1740   | 15.8             | ✅ USABLE         | -
WD2.1    | 3          | S

**LSA10 Detector Usability & Demand Selection**

This section evaluates detector usability for LD-LSA10.

The following checks are performed:

Controller failure check (from intersection definition JSON)

Zero rate analysis from September detector time-series

Demand eligibility based on the mapping table rules

Detectors are classified as:

✅ USABLE – detector produces valid measurements

❌ X (NOT USABLE) – faulty or invalid data

⚠️ Not used for demand – valid but excluded by mapping

In [ ]:
from pathlib import Path
import json
from collections import defaultdict

# =========================
# PATHS
# =========================
INTERSECTIONS_DIR = Path(
r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections"
)

SEP_BASE = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\data\Landau\2025-09")
SEP_FOLDER = SEP_BASE / "LDLSA10"


# =========================
# LSA10 mapping (from Excel)
# =========================

LSA10_MAPPING = [

("D25",5,"Straight+Right","yes but only D25"),

("VD11",1,"Straight+Right","no"),

("WD52",31,"All directions","yes but only WD52"),

("WD51",30,"All directions","no"),

("VD42",4,"Straight","yes"),

("VD41",3,"Straight+Right","yes"),

("WD32",29,"Right","yes but only WD32"),

("WD31",28,"Right","no"),

("VD25",2,"Straight","yes but only VD25"),

("VD23",26,"Straight+Right","no"),

("WD24",27,"Left","no")

]


# =========================
# Load intersection state
# =========================

def load_tlc(alias):

    for f in INTERSECTIONS_DIR.glob("*.json"):

        j=json.load(open(f,encoding="utf8"))

        tlc=j.get("tlc",{})

        if tlc.get("alias")==alias:

            return tlc

    return None


tlc=load_tlc("LD-LSA10")


fault_ids=set()

if tlc:

    for d in tlc.get("hardwareStatus",{}).get("faultDetectors",[]):

        fault_ids.add(int(d["id"]))


# =========================
# Collect detector stats
# =========================

stats=defaultdict(lambda:{"frames":0,"zero":0})

for f in SEP_FOLDER.glob("*.json"):

    j=json.load(open(f,encoding="utf8"))

    for tf in j.get("timeFrames",[]):

        for d in tf.get("detectors",[]):

            did=int(d["id"])

            c=d["reading"]["count"]["value"]

            stats[did]["frames"]+=1

            if c==0:

                stats[did]["zero"]+=1


# =========================
# Build usability table
# =========================

print("\n==============================")

print("LSA10 DETECTOR USABILITY")

print("==============================")

print("Detector | ID | Movement | ZeroRate% | Status | Reason")

print("-"*90)


for name,did,mov,demand_rule in LSA10_MAPPING:

    s=stats.get(did,{"frames":0,"zero":0})

    frames=s["frames"]

    zero=s["zero"]

    zr=(zero/frames*100) if frames else 0


    status="USABLE"

    reason="-"


    if did in fault_ids:

        status="❌ X"

        reason="controller failure"


    elif frames==0:

        status="❌ X"

        reason="no data"


    elif zr>=99:

        status="❌ X"

        reason="always zero"


    elif "only" in demand_rule and name not in demand_rule:

        status="⚠️ not used"

        reason="excluded by mapping rule"


    print(f"{name:7} | {did:3} | {mov:15} | {zr:8.1f} | {status:10} | {reason}")


LSA10 DETECTOR USABILITY
Detector | ID | Movement | ZeroRate% | Status | Reason
------------------------------------------------------------------------------------------
D25     |   5 | Straight+Right  |      0.4 | USABLE     | -
VD11    |   1 | Straight+Right  |      0.6 | USABLE     | -
WD52    |  31 | All directions  |     13.5 | USABLE     | -
WD51    |  30 | All directions  |     16.0 | USABLE     | -
VD42    |   4 | Straight        |      2.9 | USABLE     | -
VD41    |   3 | Straight+Right  |      1.8 | USABLE     | -
WD32    |  29 | Right           |     24.7 | USABLE     | -
WD31    |  28 | Right           |     14.4 | USABLE     | -
VD25    |   5 | Straight        |      0.4 | USABLE     | -
VD23    |  26 | Straight+Right  |     20.2 | USABLE     | -
WD24    |  27 | Left            |      5.9 | USABLE     | -


**LSA8 Detector Usability & Demand Selection**

This section evaluates the detectors for LD-LSA8 using:

intersection controller status (fault detectors)

September detector measurements

mapping table constraints

Detectors are classified as:

Status	Meaning
✅ USABLE	detector produces valid measurements

❌ X (NOT USABLE)	faulty or always zero

⚠️ Not used for demand	valid but excluded by mapping rule

In [3]:
from pathlib import Path
import json
from collections import defaultdict

# =========================
# PATHS
# =========================

INTERSECTIONS_DIR = Path(
r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections"
)

SEP_BASE = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\data\Landau\2025-09")

SEP_FOLDER = SEP_BASE / "LDLSA8"


# =========================
# LSA8 mapping (Excel)
# =========================

LSA8_MAPPING = [

("D61",5,"Left","yes"),

("D51",4,"Right+Straight+Left","yes"),

("D41",3,"Left","yes"),

("D21",1,"Straight+Right","yes"),

("D23",2,"Left","yes")

]


# =========================
# load controller state
# =========================

def load_tlc(alias):

    for f in INTERSECTIONS_DIR.glob("*.json"):

        j=json.load(open(f,encoding="utf8"))

        tlc=j.get("tlc",{})

        if tlc.get("alias")==alias:

            return tlc

    return None


tlc=load_tlc("LD-LSA8")

fault_ids=set()

if tlc:

    for d in tlc.get("hardwareStatus",{}).get("faultDetectors",[]):

        fault_ids.add(int(d["id"]))


# =========================
# collect detector stats
# =========================

stats=defaultdict(lambda:{"frames":0,"zero":0})

for f in SEP_FOLDER.glob("*.json"):

    j=json.load(open(f,encoding="utf8"))

    for tf in j.get("timeFrames",[]):

        for d in tf.get("detectors",[]):

            did=int(d["id"])

            c=d["reading"]["count"]["value"]

            stats[did]["frames"]+=1

            if c==0:

                stats[did]["zero"]+=1


# =========================
# build usability table
# =========================

print("\n==============================")

print("LSA8 DETECTOR USABILITY")

print("==============================")

print("Detector | ID | Movement | ZeroRate% | Status | Reason")

print("-"*90)


for name,did,mov,demand in LSA8_MAPPING:

    s=stats.get(did,{"frames":0,"zero":0})

    frames=s["frames"]

    zero=s["zero"]

    zr=(zero/frames*100) if frames else 0


    status="USABLE"

    reason="-"


    if did in fault_ids:

        status="❌ X"

        reason="controller failure"


    elif frames==0:

        status="❌ X"

        reason="no data"


    elif zr>=99:

        status="❌ X"

        reason="always zero"


    print(f"{name:6} | {did:3} | {mov:20} | {zr:8.1f} | {status:10} | {reason}")


LSA8 DETECTOR USABILITY
Detector | ID | Movement | ZeroRate% | Status | Reason
------------------------------------------------------------------------------------------
D61    |   5 | Left                 |      0.0 | ❌ X        | no data
D51    |   4 | Right+Straight+Left  |      0.0 | ❌ X        | no data
D41    |   3 | Left                 |      0.0 | ❌ X        | no data
D21    |   1 | Straight+Right       |      0.0 | ❌ X        | no data
D23    |   2 | Left                 |      0.0 | ❌ X        | no data


**LSA1 Detector Usability (movement + zero rate + failure)**

This section evaluates detectors for LD-LSA1 using:

Controller-reported failures (hardwareStatus.faultDetectors)

September time-series readings (zero rate of count)

Outputs:

A table with mapping fields (Detector, ID, Movement, Demand, edge_relation)

Zero-rate statistics from September data

USABLE / NOT USABLE (X) decision with reason

Explanation of what zero-rate means, and how usable detectors are used for demand / calibration

In [1]:
from pathlib import Path
import json
from collections import defaultdict

# =========================
# PATHS
# =========================
INTERSECTIONS_DIR = Path(
    r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections"
)

SEP_BASE = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\data\Landau\2025-09")
SEP_FOLDER_LSA1 = SEP_BASE / "LDLSA1"   # September folder naming (LDLSA1)

ALMOST_ZERO_THRESHOLD_PCT = 99.0

# =========================
# LSA1 mapping (from your screenshot)
# Columns: Detector_Name, Detector_ID, Movement, Demand, edge_relation
# =========================
LSA1_MAPPING = [
    # West (Reinstraße)
    ("DET12", 8,  "Left",              "yes", ""),
    ("DET22", 10, "Right+Straight",    "yes", ""),
    ("DET11", 7,  "Left",              "no",  "yes"),
    ("DET21", 9,  "Straight+Right",    "no",  "yes"),

    # South (Paul-von-Denis-Straße)
    ("DET32", 12, "Straight-entry",    "yes", ""),
    ("DET31", 11, "Straight",          "no",  "yes"),

    # East (Queichheimer Straße)
    ("DET42", 14, "Straight+Right",    "yes", ""),
    ("DET44", 24, "Straight",          "yes", ""),
    ("DET52", 26, "Left",              "yes", ""),
    ("DET61", 27, "Right-only",        "no",  "yes"),
    ("DET41", 13, "Straight1",         "no",  "yes"),
    ("DET43", 23, "Straight2",         "no",  "yes"),
    ("DET51", 25, "Left-only",         "no",  "yes"),

    # North (Maximilianstraße)
    ("DET74", 39, "Left-only",         "yes", ""),
    ("DET72", 29, "Straight+Right",    "yes", ""),
    ("DET73", 30, "Left-only",         "no",  "yes"),
    ("DET71", 28, "Straight-only",     "no",  "yes"),
    ("DET81", 40, "Right-only",        "no",  "yes"),
]

# =========================
# Helpers (same as before)
# =========================
def list_json_files(folder: Path):
    return sorted([p for p in folder.glob("*.json") if p.is_file()])

def load_tlc_by_alias(intersections_dir: Path, alias_target: str):
    for fp in list_json_files(intersections_dir):
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
        tlc = data.get("tlc", {})
        if tlc.get("alias") == alias_target or tlc.get("name") == alias_target:
            return tlc, fp
    return None, None

def extract_fault_detectors(tlc: dict):
    hw = tlc.get("hardwareStatus", {})
    faults = []
    for fd in hw.get("faultDetectors", []) or []:
        fd_id = fd.get("id")
        fd_name = fd.get("name")
        if fd_id is None:
            continue
        faults.append((int(fd_id), fd_name))
    uniq = {}
    for d_id, d_name in faults:
        uniq[d_id] = (d_id, d_name)
    return [uniq[k] for k in sorted(uniq.keys())]

def safe_get_count(det_obj):
    reading = det_obj.get("reading", {})
    c = reading.get("count", {})
    return c.get("value")

def pct(z, n):
    return 0.0 if n == 0 else 100.0 * z / n

# =========================
# A) Controller failure check
# =========================
tlc, fp = load_tlc_by_alias(INTERSECTIONS_DIR, "LD-LSA1")
fault_ids = set()
fault_list = []

print("\n==============================")
print("LSA1 CONTROLLER FAILURE CHECK")
print("==============================")

if tlc is None:
    print("WARNING: Could not find LD-LSA1 in intersections folder. Failure check skipped.")
else:
    fault_list = extract_fault_detectors(tlc)
    fault_ids = {d_id for d_id, _ in fault_list}

    print("Intersection file:", fp)
    print("active:", tlc.get("active"))
    print("state:", tlc.get("state"))
    print("brokenDetectorCount:", tlc.get("brokenDetectorCount"))
    print("lastStatusUpdate:", tlc.get("lastStatusUpdate"))

print("\nController-reported faulty detectors:")
if not fault_list:
    print("(none)")
else:
    for d_id, d_name in fault_list:
        print(f"- ID={d_id} | Name={d_name}")

# =========================
# B) Time-series zero-rate analysis (September)
# =========================
files = list_json_files(SEP_FOLDER_LSA1)
print("\n==============================")
print("LSA1 SEPTEMBER FILES CHECK")
print("==============================")
print("Folder:", SEP_FOLDER_LSA1)
print("Files found:", len(files))

stats = defaultdict(lambda: {"frames": 0, "count_zero": 0, "count_nonzero": 0})

for fp_ts in files:
    with open(fp_ts, "r", encoding="utf-8") as f:
        chunk = json.load(f)

    for tf in chunk.get("timeFrames", []):
        for det in tf.get("detectors", []):
            det_id = det.get("id")
            if det_id is None:
                continue
            det_id = int(det_id)

            c = safe_get_count(det)
            if not isinstance(c, (int, float)):
                continue

            stats[det_id]["frames"] += 1
            if c == 0:
                stats[det_id]["count_zero"] += 1
            else:
                stats[det_id]["count_nonzero"] += 1

# =========================
# C) Build mapping table + usability decision
# =========================
rows = []
for name, det_id, movement, demand_flag, edge_rel in LSA1_MAPPING:
    s = stats.get(det_id, {"frames": 0, "count_zero": 0, "count_nonzero": 0})
    n = s["frames"]
    cz = s["count_zero"]
    zr = pct(cz, n)

    reasons = []
    usable = True

    if det_id in fault_ids:
        usable = False
        reasons.append("controller faultDetectors")

    if n == 0:
        usable = False
        reasons.append("no timeFrames found in September data")
    else:
        if cz == n and s["count_nonzero"] == 0:
            usable = False
            reasons.append("count always 0 (100%)")
        elif zr >= ALMOST_ZERO_THRESHOLD_PCT:
            usable = False
            reasons.append(f"count almost always 0 (>= {ALMOST_ZERO_THRESHOLD_PCT}%)")

    status = "✅ USABLE" if usable else "❌ X (NOT USABLE)"
    reason_text = " / ".join(reasons) if reasons else "-"

    rows.append({
        "Detector": name,
        "Detector_ID": det_id,
        "Movement": movement,
        "Demand": demand_flag,
        "Edge_relation": edge_rel,
        "Frames": n,
        "Zero_rate_% (count)": round(zr, 1),
        "Status": status,
        "Reason": reason_text
    })

print("\n==============================")
print("LSA1 DETECTOR USABILITY TABLE")
print("==============================")
header = ["Detector","Detector_ID","Movement","Demand","Edge_relation","Frames","Zero_rate_% (count)","Status","Reason"]
print(" | ".join(header))
print("-" * 160)

for r in rows:
    print(
        f"{r['Detector']:<7} | {r['Detector_ID']:<10} | {r['Movement']:<16} | {r['Demand']:<6} | {r['Edge_relation']:<12} | "
        f"{r['Frames']:<6} | {r['Zero_rate_% (count)']:<16} | {r['Status']:<16} | {r['Reason']}"
    )

# =========================
# D) Explain zero rate and usage note
# =========================
print("\n==============================")
print("What does 'Zero rate' mean?")
print("==============================")
print(
    "Zero rate (count) = (# of timeFrames where detector count == 0) / (total # of timeFrames for that detector) * 100.\n"
    "Interpretation:\n"
    "- Low zero rate can be normal (e.g., at night / low demand).\n"
    "- Very high zero rate (near 100%) typically indicates a broken detector, wrong mapping, or missing data.\n"
    f"- In this notebook, detectors with zero rate >= {ALMOST_ZERO_THRESHOLD_PCT}% (or exactly 100%) are marked NOT USABLE."
)

print("\n==============================")
print("Note for demand generation / calibration")
print("==============================")
print(
    "For corridor calibration, LSA1 is typically used as an OUTFLOW target.\n"
    "We prefer using detectors marked Demand=yes AND USABLE as measurement targets.\n\n"
    "Detectors marked as '❌ X (NOT USABLE)' must be excluded from:\n"
    "- calibration targets (GEH/RMSE comparison)\n"
    "- demand generation inputs\n"
    "- LSTM features\n\n"
    "Detectors with Demand=no but USABLE can still be used for:\n"
    "- movement plausibility checks\n"
    "- consistency checks (edge_relation)."
)


LSA1 CONTROLLER FAILURE CHECK
Intersection file: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections\LSA1.json
active: True
state: INTERNAL_FAILURE
brokenDetectorCount: 2
lastStatusUpdate: 2025-06-04T09:30:00

Controller-reported faulty detectors:
- ID=25 | Name=DET51
- ID=29 | Name=DET72

LSA1 SEPTEMBER FILES CHECK
Folder: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\data\Landau\2025-09\LDLSA1
Files found: 87

LSA1 DETECTOR USABILITY TABLE
Detector | Detector_ID | Movement | Demand | Edge_relation | Frames | Zero_rate_% (count) | Status | Reason
----------------------------------------------------------------------------------------------------------------------------------------------------------------
DET12   | 8          | Left             | yes    |              | 1740   | 0.2              | ✅ USABLE         | -
DET22   | 10         | Right+Straight   | yes    |              | 1740   | 0.2              | ✅ USABLE     